In [1]:
import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
)
OUTPUT_DIR = "./checkpoint-paircls"
# 1. KLUE-NLI 데이터셋 로드
dataset = load_dataset(
    "parquet",
    data_files={
        "train": (
            "https://huggingface.co/datasets/klue/klue/resolve/"
            "refs%2Fconvert%2Fparquet/nli/train/0000.parquet"
        ),
        "validation": (
            "https://huggingface.co/datasets/klue/klue/resolve/"
            "refs%2Fconvert%2Fparquet/nli/validation/0000.parquet"
        ),
    },
)

print(dataset)
print(dataset["train"].column_names)

# 2. 토크나이저 및 모델 로드 (KcBERT-base, NLI는 3개의 레이블: entailment, neutral, contradiction)
model_name = "beomi/kcbert-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=3)

# 3. 데이터셋 토큰화 함수 정의 (premise와 hypothesis 쌍 입력)
def tokenize_function(examples):
    return tokenizer(
        examples["premise"],
        examples["hypothesis"],
        max_length=64,
        padding="max_length",
        truncation=True,
    )

# 데이터셋 전체에 토큰화 적용
tokenized_datasets = dataset.map(tokenize_function, batched=True)

# 4. 학습 설정 (TrainingArguments)
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=32 if torch.cuda.is_available() else 4,
    per_device_eval_batch_size=32 if torch.cuda.is_available() else 4,
    learning_rate=5e-5,
    num_train_epochs=5,
    seed=7,
    eval_strategy="epoch",  # 에포크마다 평가
    save_strategy="epoch",  # 에포크마다 모델 저장
    load_best_model_at_end=True,
    logging_steps=100,
)

# 5. Trainer 초기화 및 학습 실행
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    processing_class=tokenizer,
)

trainer.train()
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

f:\Project\AI-course\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


DatasetDict({
    train: Dataset({
        features: ['guid', 'source', 'premise', 'hypothesis', 'label'],
        num_rows: 24998
    })
    validation: Dataset({
        features: ['guid', 'source', 'premise', 'hypothesis', 'label'],
        num_rows: 3000
    })
})
['guid', 'source', 'premise', 'hypothesis', 'label']


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1005.28it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: beomi/kcbert-base
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoi

Epoch,Training Loss,Validation Loss


KeyboardInterrupt: 

In [15]:
print(trainer.state.best_model_checkpoint)
print(trainer.state.best_metric)
print(trainer.evaluate())

./checkpoint-paircls\checkpoint-1564
0.838615894317627


Training Loss,Validation Loss,Epoch
0.090017,0.838616,5


{'eval_loss': 0.838615894317627}


In [16]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# 1. 저장된 모델과 토크나이저 로드
MODEL_DIR = "./checkpoint-paircls"
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = AutoModelForSequenceClassification.from_pretrained(MODEL_DIR)
model.to(device)

# 평가 모드 전환
model.eval()

# 2. 추론할 문장 쌍 준비
#premise = "국내 야구 리그에서 새로운 기록이 세워졌다."
#hypothesis = "어제 프로야구 경기에서 신기록이 나왔다."
premise = "나는 맥북을 쓰지 않았다."
hypothesis = "나는 맥북을 썼다."

# 3. 토크나이징 (학습 때와 동일한 방식으로 묶어서 입력)
inputs = tokenizer(
    premise,
    hypothesis,
    max_length=64,
    padding="max_length",
    truncation=True,
    return_tensors="pt" # PyTorch 텐서 형태로 반환
)
inputs = {key: value.to(device) for key, value in inputs.items()}
# 4. 모델 예측 수행
with torch.no_grad():
    outputs = model(**inputs)
    logits = outputs.logits
    # 가장 확률이 높은 클래스 인덱스 추출
    pred_label_id = torch.argmax(logits, dim=-1).item()

# 5. KLUE-NLI 레이블 맵핑 (0: entailment, 1: neutral, 2: contradiction)
label_map = {0: "entailment (함의)", 1: "neutral (중립)", 2: "contradiction (모순)"}

print(f"전제: {premise}")
print(f"가설: {hypothesis}")
print(f"예측 결과: {label_map[pred_label_id]}")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

전제: 나는 맥북을 쓰지 않았다.
가설: 나는 맥북을 썼다.
예측 결과: entailment (함의)
